# RAON Pipeline — Interactive Inference

This notebook demonstrates how to use `RaonPipeline` for all supported task types:

| Task | Input | Output |
|------|-------|--------|
| **STT** | Audio | Text |
| **TTS** | Text, Audio (optional, speaker ref) | Audio |
| **TextQA** | Text, Audio (optional) | Text |
| **SpeechChat** | Audio | Text |
| **Chat** | Text, Audio (free-form) | Text / Audio |

Messages follow a multimodal format: a list of turns, where each turn's `content` is either a plain string or a list of typed parts (`{"type": "text", ...}`, `{"type": "audio", ...}`).

In [ ]:
# Requirements:
#   pip install transformers>=4.57.1 torch torchaudio soundfile accelerate
#
# For local model loading, also install raon:
#   pip install -e .   (or: uv sync)
#
# Optional: install FlashAttention 2 for lower memory usage
#   pip install -U flash-attn --no-build-isolation

## Model Loading

Choose **one** of the two options below:
- **Option A**: Load from HuggingFace Hub (no `pip install raon` needed)
- **Option B**: Load from a local checkpoint path (requires `pip install -e .`)

### Option A: Load from HuggingFace Hub

No `pip install raon` needed — only `transformers`, `torch`, `torchaudio`, `soundfile`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import torch
from IPython.display import Audio, display
from transformers import AutoConfig
from transformers.dynamic_module_utils import get_class_from_dynamic_module

# ── Configuration ──────────────────────────────────────────────────────────
MODEL_ID = "KRAFTON/Raon-Speech-9B"
DEVICE = "cuda"
DTYPE = "bfloat16"
OUTPUT_DIR = Path("output/notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Resolve Hub module
print(f"Resolving Hub module from {MODEL_ID}...")
_cfg = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
_revision = getattr(_cfg, "_commit_hash", None)
RaonPipeline = get_class_from_dynamic_module(
    "modeling_raon.RaonPipeline",
    MODEL_ID,
    revision=_revision,
)
del _cfg, _revision

# Create pipeline
print("Creating RaonPipeline...")
pipe = RaonPipeline(MODEL_ID, device=DEVICE, dtype=DTYPE)
print("Pipeline ready!")

### Option B: Load from Local Path

Requires `pip install -e .` (or `uv sync`) from the repository root.

In [ ]:
from __future__ import annotations

from pathlib import Path

from IPython.display import Audio, display

from raon import RaonPipeline

# ── Configuration ──────────────────────────────────────────────────────────
MODEL_PATH = "/path/to/raon-model"  # <-- set this
DEVICE = "cuda"
DTYPE = "bfloat16"
OUTPUT_DIR = Path("output/notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load the pipeline
print(f"Loading RaonPipeline from {MODEL_PATH} ...")
pipe = RaonPipeline(MODEL_PATH, device=DEVICE, dtype=DTYPE)
print("Pipeline ready.")

---

## Audio Paths

Set example audio paths (relative to `examples/`).

In [ ]:
AUDIO_DIR = Path("../data/speechllm/eval/audio")
DUMMY_STT_AUDIO = str(AUDIO_DIR / "stt.wav")
DUMMY_STT_AUDIO_KOR = str(AUDIO_DIR / "stt_kor.wav")
DUMMY_TTS_SPEAKER = str(AUDIO_DIR / "spk_ref.wav")
DUMMY_TTS_SPEAKER_KOR = str(AUDIO_DIR / "spk_ref_kor.wav")
DUMMY_TEXTQA_AUDIO = str(AUDIO_DIR / "textqa.wav")
DUMMY_TEXTQA_AUDIO_KOR = str(AUDIO_DIR / "textqa_kor.wav")
DUMMY_SPEECH_CHAT = str(AUDIO_DIR / "speech-chat.wav")
DUMMY_SPEECH_CHAT_KOR = str(AUDIO_DIR / "speech-chat_kor.wav")

## STT (audio → text)

Transcribe an audio file.

In [ ]:
transcript = pipe.stt(DUMMY_STT_AUDIO)
transcript_kor = pipe.stt(DUMMY_STT_AUDIO_KOR)
print("English transcript:", transcript)
print("Korean transcript:", transcript_kor)

(OUTPUT_DIR / "stt_output.txt").write_text(transcript, encoding="utf-8")
(OUTPUT_DIR / "stt_output_kor.txt").write_text(transcript_kor, encoding="utf-8");

## TTS (text → audio)

Synthesize speech from text. Optionally provide a speaker reference audio for voice conditioning.

In [ ]:
tts_text = "Hello, how are you today?"
tts_text_kor = "아니야 그거. 그냥 환불해 오빠. 환불하는 게 훨씬 우리한테 이득이야."

audio, sr = pipe.tts(tts_text, speaker_audio=DUMMY_TTS_SPEAKER)
audio_kor, sr_kor = pipe.tts(tts_text_kor, speaker_audio=DUMMY_TTS_SPEAKER_KOR)

out_path = OUTPUT_DIR / "tts_output.wav"
out_path_kor = OUTPUT_DIR / "tts_output_kor.wav"
pipe.save_audio((audio, sr), str(out_path))
pipe.save_audio((audio_kor, sr_kor), str(out_path_kor))
print(f"Saved English audio to {out_path} ({audio.shape[0] / sr:.2f}s)")
print(f"Saved Korean audio to {out_path_kor} ({audio_kor.shape[0] / sr_kor:.2f}s)")

display(Audio(audio.float().cpu().numpy(), rate=sr))
display(Audio(audio_kor.float().cpu().numpy(), rate=sr_kor))

## TextQA (text + optional audio → text)

Ask a question with optional audio context. When audio is provided, the model uses it as context for the answer.

In [ ]:
question = "What is the gender of the speaker?"
question_kor = "화자가 느끼는 자신의 성격 관련 고민은 무엇인가요?"

answer = pipe.textqa(question, audio=DUMMY_TEXTQA_AUDIO)
answer_kor = pipe.textqa(question_kor, audio=DUMMY_TEXTQA_AUDIO_KOR)
print("English question:", question)
print("English answer:", answer)
print("Korean question:", question_kor)
print("Korean answer:", answer_kor)

(OUTPUT_DIR / "textqa_output.txt").write_text(answer, encoding="utf-8")
(OUTPUT_DIR / "textqa_output_kor.txt").write_text(answer_kor, encoding="utf-8");

## SpeechChat (audio → text)

The question is spoken as audio. The model listens and answers in text.

In [ ]:
answer = pipe.speech_chat(DUMMY_SPEECH_CHAT)
answer_kor = pipe.speech_chat(DUMMY_SPEECH_CHAT_KOR)
print("English answer:", answer)
print("Korean answer:", answer_kor)

(OUTPUT_DIR / "speech_chat_output.txt").write_text(answer, encoding="utf-8")
(OUTPUT_DIR / "speech_chat_output_kor.txt").write_text(answer_kor, encoding="utf-8");

## Chat (text + audio, free-form → text / audio)

Use `pipe.chat()` to pass arbitrary multimodal conversations directly — full control over role, system prompt, and mixed text/audio content.

Each message is a dict with `role` and `content`.  
`content` can be a plain string **or** a list of typed parts:
- `{"type": "text", "text": "..."}` — text
- `{"type": "audio", "audio": "/path/to/file.wav"}` — audio file

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": DUMMY_STT_AUDIO},
            {"type": "text", "text": "Please transcribe this audio and summarise the content."},
        ],
    },
]
messages_kor = [
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": DUMMY_STT_AUDIO_KOR},
            {"type": "text", "text": "이 오디오를 받아쓰고 핵심 내용을 짧게 요약해 주세요."},
        ],
    },
]

response = pipe.chat(messages)
response_kor = pipe.chat(messages_kor)
print("English response:", response)
print("Korean response:", response_kor)

(OUTPUT_DIR / "chat_output.txt").write_text(response, encoding="utf-8")
(OUTPUT_DIR / "chat_output_kor.txt").write_text(response_kor, encoding="utf-8");

## Cleanup

In [ ]:
del pipe
import gc
gc.collect()
torch.cuda.empty_cache()
print("Cleaned up.")